<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>Embeddings</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import voyageai
import anthropic
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: 87485bac20625f43da5809d38ec119aca4321f71

IPython   : 9.8.0
voyageai  : 0.3.5
sklearn   : 1.7.2
anthropic : 0.75.0
matplotlib: 3.10.7
numpy     : 2.3.5
watermark : 2.5.0



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Setup and Imports


Initialize clients. Both use environment variables: `VOYAGE_API_KEY` and `ANTHROPIC_API_KEY`.

In [4]:
vo = voyageai.Client()
claude = anthropic.Anthropic()

Generating our first embedding

In [5]:
text = "Machine learning is a subset of artificial intelligence."

result = vo.embed(
    texts=[text],
    model="voyage-2",  # General-purpose embedding model
    input_type="document"
)

In [6]:
embedding = result.embeddings[0]

In [7]:
print(f"Text: '{text}'")
print(f"\nEmbedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"\nEmbedding type: {type(embedding)}")

Text: 'Machine learning is a subset of artificial intelligence.'

Embedding dimensions: 1024
First 10 values: [-0.011315016075968742, 0.02605654112994671, 0.0277328509837389, 0.044619929045438766, -0.07662053406238556, -0.057446401566267014, -0.002389053115621209, 0.00237516057677567, -0.043616801500320435, 0.04649369418621063]

Embedding type: <class 'list'>


## Semantic Similarity

Similar texts should have similar embeddings (high cosine similarity)

In [8]:
texts = [
    "The cat sat on the mat.",
    "A kitten was resting on the rug.",
    "Python is a programming language.",
    "Dogs are loyal pets.",
    "Machine learning models process data."
]

Generate embeddings for all texts

In [9]:
result = vo.embed(
    texts=texts,
    model="voyage-2",
    input_type="document"
)

embeddings = np.array(result.embeddings)

Calculate cosine similarity matrix

In [10]:
similarity_matrix = cosine_similarity(embeddings)

# Display results
print("Texts:")
for i, t in enumerate(texts):
    print(f"  {i}: {t}")

print("\nCosine Similarity Matrix:")
print("     ", end="")
for i in range(len(texts)):
    print(f"  {i}   ", end="")
print()

for i in range(len(texts)):
    print(f"  {i}: ", end="")
    for j in range(len(texts)):
        print(f"{similarity_matrix[i][j]:.3f} ", end="")
    print()

Texts:
  0: The cat sat on the mat.
  1: A kitten was resting on the rug.
  2: Python is a programming language.
  3: Dogs are loyal pets.
  4: Machine learning models process data.

Cosine Similarity Matrix:
       0     1     2     3     4   
  0: 1.000 0.930 0.819 0.852 0.820 
  1: 0.930 1.000 0.804 0.848 0.788 
  2: 0.819 0.804 1.000 0.838 0.844 
  3: 0.852 0.848 0.838 1.000 0.820 
  4: 0.820 0.788 0.844 0.820 1.000 


## Question Answering with Semantic Search

We start by defining a sample "knowledge base" about Python

In [11]:
knowledge_base = [
    {
        "id": 1,
        "title": "Python Basics",
        "content": "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasizes code readability with its notable use of significant indentation."
    },
    {
        "id": 2,
        "title": "Python Data Types",
        "content": "Python has several built-in data types including integers, floats, strings, lists, tuples, dictionaries, and sets. Variables in Python are dynamically typed, meaning you don't need to declare their type."
    },
    {
        "id": 3,
        "title": "Python Functions",
        "content": "Functions in Python are defined using the 'def' keyword. They can accept parameters, return values, and support default arguments, keyword arguments, and variable-length arguments with *args and **kwargs."
    },
    {
        "id": 4,
        "title": "Python Classes",
        "content": "Python supports object-oriented programming with classes. Classes are defined using the 'class' keyword and can have methods, attributes, inheritance, and special methods like __init__ and __str__."
    },
    {
        "id": 5,
        "title": "Python List Comprehensions",
        "content": "List comprehensions provide a concise way to create lists. The syntax is [expression for item in iterable if condition]. They are more readable and often faster than traditional loops."
    },
    {
        "id": 6,
        "title": "Python Virtual Environments",
        "content": "Virtual environments isolate Python dependencies for different projects. Tools like venv, virtualenv, and conda help manage these environments. They prevent conflicts between package versions."
    },
    {
        "id": 7,
        "title": "Python Package Management",
        "content": "pip is Python's package installer. It installs packages from PyPI (Python Package Index). requirements.txt files list project dependencies. Modern alternatives include Poetry and uv."
    },
    {
        "id": 8,
        "title": "Python Error Handling",
        "content": "Python uses try/except blocks for error handling. Common exceptions include ValueError, TypeError, KeyError, and FileNotFoundError. You can also create custom exceptions by inheriting from Exception."
    }
]

print(f"Knowledge base contains {len(knowledge_base)} documents")

Knowledge base contains 8 documents


Generate embeddings

In [12]:
doc_texts = [doc["content"] for doc in knowledge_base]

doc_result = vo.embed(
    texts=doc_texts,
    model="voyage-2",
    input_type="document"
)

doc_embeddings = np.array(doc_result.embeddings)

Now we can find the most relevant documents for a query using semantic search.

In [13]:
def semantic_search(query, doc_embeddings, knowledge_base, top_k=3):
    # Query embedding
    query_result = vo.embed(
        texts=[query],
        model="voyage-2",
        input_type="query"  # Note: use "query" for search queries
    )
    
    query_embedding = np.array(query_result.embeddings[0])
    
    # Cosine similarity with every documents
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    
    # Get top-k most similar documents
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "document": knowledge_base[idx],
            "similarity": similarities[idx]
        })
    
    return results

Test the search

In [14]:
query = "How do I handle errors in Python?"
results = semantic_search(query, doc_embeddings, knowledge_base)

print(f"Query: '{query}'")
print("\nTop relevant documents:\n")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['document']['title']} (similarity: {r['similarity']:.3f})")
    print(f"   {r['document']['content'][:100]}...")
    print()

Query: 'How do I handle errors in Python?'

Top relevant documents:

1. Python Error Handling (similarity: 0.787)
   Python uses try/except blocks for error handling. Common exceptions include ValueError, TypeError, K...

2. Python Functions (similarity: 0.705)
   Functions in Python are defined using the 'def' keyword. They can accept parameters, return values, ...

3. Python Data Types (similarity: 0.705)
   Python has several built-in data types including integers, floats, strings, lists, tuples, dictionar...



## Question Answering with RAG

Three step process:
1. Identify relevant documents via semantic search
2. Build context from results
3. Generate an answer based on the context

In [15]:
def answer_question(query, doc_embeddings, knowledge_base, claude_client, top_k=3):
    # Step 1: Retrieve relevant documents
    results = semantic_search(query, doc_embeddings, knowledge_base, top_k)
    
    # Step 2: Build context from retrieved documents
    context = "\n\n".join([
        f"Document: {r['document']['title']}\n{r['document']['content']}"
        for r in results
    ])
    
    # Step 3: Generate answer with Claude
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        system="""You are a helpful Python tutor. Answer questions based on the provided context.
If the context doesn't contain enough information, say so.
Be concise and provide code examples when helpful.""",
        messages=[{
            "role": "user",
            "content": f"""
<context>
{context}
</context>

<question>
{query}
</question>

Please answer the question based on the context provided.
"""
        }]
    )
    
    return {
        "answer": response.content[0].text,
        "sources": [r["document"]["title"] for r in results]
    }

A simple example:

In [16]:
query = "How do I create a function with default parameters?"
result = answer_question(query, doc_embeddings, knowledge_base, claude)

print(f"Question: {query}")
print(f"\nSources: {', '.join(result['sources'])}")
print(f"\nAnswer:\n{result['answer']}")

Question: How do I create a function with default parameters?

Sources: Python Functions, Python Data Types, Python Error Handling

Answer:
Based on the context provided, Python functions support default arguments. Here's how you create a function with default parameters:

You define default parameters by assigning values to parameters in the function definition using the `def` keyword. Here's the syntax:

```python
def function_name(param1, param2=default_value, param3=another_default):
    # function body
    return result
```

**Example:**
```python
def greet(name, message="Hello", punctuation="!"):
    return f"{message} {name}{punctuation}"

# Usage examples:
print(greet("Alice"))                    # Output: Hello Alice!
print(greet("Bob", "Hi"))               # Output: Hi Bob!
print(greet("Charlie", "Hey", "?"))     # Output: Hey Charlie?
```

**Key points:**
- Parameters with default values must come after parameters without defaults
- You can call the function with fewer argum

Try more questions

In [19]:
question = "What are the different data types in Python?"

result = answer_question(question, doc_embeddings, knowledge_base, claude)

print(f"Question: {question}")
print(f"Sources: {', '.join(result['sources'])}")
print(f"Result: {result['answer'][:200]}...")

Question: What are the different data types in Python?
Sources: Python Data Types, Python Classes, Python Functions
Result: Based on the context provided, Python has several built-in data types:

- **Integers** - whole numbers
- **Floats** - decimal numbers  
- **Strings** - text data
- **Lists** - ordered, mutable collect...


## Recommendations

Embeddings can power recommendation systems by finding similar items.


A simple product catalog

In [20]:
products = [
    {"id": 0, "name": "Laptop Stand", "description": "Ergonomic aluminum laptop stand with adjustable height for better posture while working."},
    {"id": 1, "name": "Mechanical Keyboard", "description": "RGB mechanical keyboard with Cherry MX switches for comfortable typing and gaming."},
    {"id": 2, "name": "Wireless Mouse", "description": "Ergonomic wireless mouse with precision tracking and programmable buttons."},
    {"id": 3, "name": "Monitor Arm", "description": "Adjustable monitor arm that clamps to desk, frees up space and improves viewing angle."},
    {"id": 4, "name": "USB Hub", "description": "7-port USB 3.0 hub with fast data transfer for connecting multiple devices."},
    {"id": 5, "name": "Webcam HD", "description": "1080p HD webcam with built-in microphone for video calls and streaming."},
    {"id": 6, "name": "Desk Mat", "description": "Large desk mat with smooth surface for mouse tracking and wrist comfort."},
    {"id": 7, "name": "Cable Management Kit", "description": "Complete cable organizer set with clips, ties, and sleeves to keep desk tidy."},
    {"id": 8, "name": "Ring Light", "description": "LED ring light with adjustable brightness for video calls and content creation."},
    {"id": 9, "name": "Noise-Canceling Headphones", "description": "Wireless headphones with active noise cancellation for focused work and music."}
]

Create embeddings for products

In [21]:
product_texts = [f"{p['name']}: {p['description']}" for p in products]
product_result = vo.embed(texts=product_texts, model="voyage-2", input_type="document")
product_embeddings = np.array(product_result.embeddings)

Generate recommendations based on product similarity.

In [22]:
def get_recommendations(product_id, product_embeddings, products, top_k=3):
    target_embedding = product_embeddings[product_id]
    similarities = cosine_similarity([target_embedding], product_embeddings)[0]
    
    top_indices = np.argsort(similarities)[::-1]
    
    recommendations = []
    for idx in top_indices:
        if idx != product_id and len(recommendations) < top_k:
            recommendations.append({
                "product": products[idx],
                "similarity": similarities[idx]
            })
    
    return recommendations

Get recommendation

In [23]:
product_id = 0  # Laptop Stand
recommendations = get_recommendations(product_id, product_embeddings, products)

In [24]:
print(f"Recommendations for: {products[product_id]['name']}")
print(f"   {products[product_id]['description']}")
print("\nSimilar products you might like:\n")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['product']['name']} (similarity: {rec['similarity']:.3f})")
    print(f"   {rec['product']['description']}")
    print()

Recommendations for: Laptop Stand
   Ergonomic aluminum laptop stand with adjustable height for better posture while working.

Similar products you might like:

1. Monitor Arm (similarity: 0.870)
   Adjustable monitor arm that clamps to desk, frees up space and improves viewing angle.

2. Wireless Mouse (similarity: 0.848)
   Ergonomic wireless mouse with precision tracking and programmable buttons.

3. Desk Mat (similarity: 0.840)
   Large desk mat with smooth surface for mouse tracking and wrist comfort.



Search-based recommendations

In [25]:
def search_products(query, product_embeddings, products, top_k=3):
    query_result = vo.embed(texts=[query], model="voyage-2", input_type="query")
    query_embedding = np.array(query_result.embeddings[0])
    
    similarities = cosine_similarity([query_embedding], product_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "product": products[idx],
            "similarity": similarities[idx]
        })
    return results

In [26]:
queries = [
    "I need something for video calls",
    "help me organize my desk",
    "comfortable typing experience"
]

for query in queries:
    results = search_products(query, product_embeddings, products)
    print(f"Search: '{query}'")
    for r in results:
        print(f"   • {r['product']['name']} ({r['similarity']:.3f})")
    print()

Search: 'I need something for video calls'
   • Webcam HD (0.762)
   • Ring Light (0.715)
   • Mechanical Keyboard (0.663)

Search: 'help me organize my desk'
   • Cable Management Kit (0.758)
   • Monitor Arm (0.715)
   • Desk Mat (0.706)

Search: 'comfortable typing experience'
   • Mechanical Keyboard (0.693)
   • Desk Mat (0.670)
   • Cable Management Kit (0.647)



## Handling Long Texts

Sample long document

In [27]:
long_document = """
# Complete Guide to Python Web Development

## Chapter 1: Introduction to Web Frameworks

Python offers several web frameworks for building applications. The most popular ones are 
Django and Flask. Django is a full-featured framework that includes an ORM, admin interface, 
and authentication system. Flask is a micro-framework that gives developers more control 
over component choices.

## Chapter 2: Setting Up Your Development Environment

Before starting web development, you need to set up your environment. Install Python 3.8+, 
create a virtual environment using venv or conda, and install your chosen framework. 
For Django: pip install django. For Flask: pip install flask.

## Chapter 3: Understanding HTTP and REST

Web applications communicate using HTTP protocol. REST (Representational State Transfer) 
is an architectural style for designing networked applications. RESTful APIs use HTTP 
methods: GET for reading, POST for creating, PUT for updating, DELETE for removing.

## Chapter 4: Database Integration

Most web applications need a database. Django includes built-in support for PostgreSQL, 
MySQL, SQLite, and Oracle through its ORM. Flask typically uses SQLAlchemy for database 
operations. Understanding SQL and database design is essential for web developers.

## Chapter 5: Authentication and Security

Security is crucial in web development. Implement user authentication with sessions or 
tokens (JWT). Protect against common vulnerabilities: SQL injection, XSS (Cross-Site 
Scripting), CSRF (Cross-Site Request Forgery). Always use HTTPS in production.

## Chapter 6: Deployment

Deploying Python web apps involves choosing a hosting platform (AWS, Heroku, DigitalOcean), 
setting up a production server (Gunicorn, uWSGI), configuring a reverse proxy (Nginx), 
and implementing CI/CD pipelines for automated deployment.
"""

print(f"Document length: {len(long_document)} characters")
print(f"Approximate words: {len(long_document.split())}")


Document length: 1853 characters
Approximate words: 257


We break down the document into smaller chunks

In [28]:
def chunk_document(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        # Try to end at a sentence boundary
        if end < len(text):
            # Look for sentence endings near the chunk boundary
            for boundary in ['. ', '.\n', '! ', '? ']:
                boundary_pos = text.rfind(boundary, start + chunk_size - 100, end + 50)
                if boundary_pos != -1:
                    end = boundary_pos + len(boundary)
                    break
        
        chunk = text[start:end].strip()
        if chunk:  # Only add non-empty chunks
            chunks.append({
                "text": chunk,
                "start": start,
                "end": end
            })
        
        start = end - overlap
    
    return chunks

Chunk the document

In [29]:
chunks = chunk_document(long_document, chunk_size=600, overlap=50)

print(f"Document split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk['text'])} chars")
    print(f"  Preview: {chunk['text'][:80]}...")
    print()

Document split into 4 chunks:

Chunk 1: 621 chars
  Preview: # Complete Guide to Python Web Development

## Chapter 1: Introduction to Web Fr...

Chunk 2: 644 chars
  Preview: venv or conda, and install your chosen framework. 
For Django: pip install djang...

Chunk 3: 599 chars
  Preview: pically uses SQLAlchemy for database 
operations. Understanding SQL and database...

Chunk 4: 133 chars
  Preview: production server (Gunicorn, uWSGI), configuring a reverse proxy (Nginx), 
and i...



Embed chunks

In [30]:
chunk_texts = [c["text"] for c in chunks]
chunk_result = vo.embed(texts=chunk_texts, model="voyage-2", input_type="document")
chunk_embeddings = np.array(chunk_result.embeddings)

Search for relevant chunks in the document

In [31]:
def search_chunks(query, chunk_embeddings, chunks, top_k=2):
    query_result = vo.embed(texts=[query], model="voyage-2", input_type="query")
    query_embedding = np.array(query_result.embeddings[0])
    
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "chunk_id": idx,
            "similarity": similarities[idx]
        })
    return results

In [32]:
query = "How do I secure my web application?"
results = search_chunks(query, chunk_embeddings, chunks)

In [33]:
print(f"Query: '{query}'")
print("\nRelevant sections:\n")
for r in results:
    print(f"Chunk {r['chunk_id']+1} (similarity: {r['similarity']:.3f}):")
    print(f"{r['chunk']['text']}")
    print("\n" + "-"*50 + "\n")

Query: 'How do I secure my web application?'

Relevant sections:

Chunk 3 (similarity: 0.725):
pically uses SQLAlchemy for database 
operations. Understanding SQL and database design is essential for web developers.

## Chapter 5: Authentication and Security

Security is crucial in web development. Implement user authentication with sessions or 
tokens (JWT). Protect against common vulnerabilities: SQL injection, XSS (Cross-Site 
Scripting), CSRF (Cross-Site Request Forgery). Always use HTTPS in production.

## Chapter 6: Deployment

Deploying Python web apps involves choosing a hosting platform (AWS, Heroku, DigitalOcean), 
setting up a production server (Gunicorn, uWSGI), configuring

--------------------------------------------------

Chunk 1 (similarity: 0.696):
# Complete Guide to Python Web Development

## Chapter 1: Introduction to Web Frameworks

Python offers several web frameworks for building applications. The most popular ones are 
Django and Flask. Django is a full-feature

### Complete RAG pipeline to answer questions about a long document

In [34]:
def answer_from_document(query, chunk_embeddings, chunks, claude_client, top_k=2):
    results = search_chunks(query, chunk_embeddings, chunks, top_k)

    context = "\n\n---\n\n".join([r["chunk"]["text"] for r in results])
    
    response = claude_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=400,
        system="You are a helpful assistant answering questions about a document. Use only the provided context.",
        messages=[{
            "role": "user",
            "content": f"""
<context>
{context}
</context>

<question>
{query}
</question>

Answer based on the context provided. If the context doesn't contain the answer, say so.
"""
        }]
    )
    
    return response.content[0].text

In [35]:
questions = [
    "What's the difference between Django and Flask?",
    "How do I deploy a Python web application?",
    "What security threats should I protect against?"
]

for q in questions:
    answer = answer_from_document(q, chunk_embeddings, chunks, claude)
    print(f"Question: {q}")
    print(f"Answer: {answer}")
    print("\n" + "="*60 + "\n")

Question: What's the difference between Django and Flask?
Answer: Based on the provided context, here are the key differences between Django and Flask:

**Django:**
- A full-featured framework
- Includes built-in components like:
  - ORM (Object-Relational Mapping)
  - Admin interface
  - Authentication system
- Has built-in support for multiple databases (PostgreSQL, MySQL, SQLite, and Oracle)

**Flask:**
- A micro-framework
- Gives developers more control over component choices
- Typically uses SQLAlchemy for database operations (rather than having built-in database support like Django)

In essence, Django is more comprehensive and comes with many features out of the box, while Flask is more minimal and flexible, allowing developers to choose their own components and tools.


Question: How do I deploy a Python web application?
Answer: Based on the provided context, deploying a Python web application involves several key steps:

1. **Choose a hosting platform** - The context mentions 

<center>
     <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>